# Lezione 6 — Addestrare una CNN

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife, basato sulla certificazione EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Durata stimata della lezione: 120 minuti**

## Obiettivi di apprendimento di questa lezione

- Addestrare una CNN da zero su MNIST
- Monitorare metriche di accuratezza oltre alla perdita
- Riconoscere i problemi di convergenza e come correggere learning rate e architettura
- Confrontare il tempo di addestramento e l'accuratezza di una CNN con una rete fully connected

Questo notebook accompagna il capitolo 06 del manuale del corso. Esegui ogni cella in ordine con **Maiusc + Invio**.

## Parte 1 — Preparazione: carica il dataset

Riutiliziamo MNIST con DataLoader, come nelle lezioni precedenti.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

print('Versión de PyTorch:', torch.__version__)
print('Versión de torchvision:', torchvision.__version__)

## Parte 2 — Definisci la CNN

La stessa CNN che hai visto nella lezione 5: due strati convoluzionali con pooling, poi strati fully connected per la classificazione.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

class_names = ['avión', 'automóvil', 'pájaro', 'gato', 'ciervo',
               'perro', 'rana', 'caballo', 'barco', 'camión']

print('Imágenes de entrenamiento:', len(train_dataset))
print('Imágenes de prueba:', len(test_dataset))

In [ ]:
def desnormalizar(img_tensor):
    # Revertimos la normalización (0.5, 0.5, 0.5) para poder visualizar la imagen
    img = img_tensor * 0.5 + 0.5
    return img.permute(1, 2, 0).numpy()

imagenes, etiquetas = next(iter(train_loader))

plt.figure(figsize=(9, 3))
for i in range(6):
    plt.subplot(1, 6, i + 1)
    plt.imshow(desnormalizar(imagenes[i]))
    plt.title(class_names[etiquetas[i]])
    plt.axis('off')
plt.show()

## Parte 3 — Metriche: oltre la perdita

Durante l'addestramento, tracci spesso la **perdita** (loss) per capire se il modello sta migliorando. Ma per una classificazione, è più intuitivo tracciare l'**accuratezza**: quale percentuale di previsioni sono corrette?

In [ ]:
class ConvNetCIFAR(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)    # 3 canales de entrada (RGB)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 10)  # 10 clases de CIFAR-10

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))   # 16x16 -> 8x8
        x = self.pool(F.relu(self.conv3(x)))   # 8x8 -> 4x4
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


model = ConvNetCIFAR()
print(model)

## Parte 4 — Funzione di valutazione con accuratezza

Calcola l'accuratezza su un batch: predicendo la classe con la probabilità più alta e confrontandola con l'etichetta vera.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Parte 5 — Il ciclo di addestramento con metriche

Addestrarai la CNN per 10-20 epoch, tracciando la perdita e l'accuratezza su entrambi gli insiemi (addestramento e validazione).

In [ ]:
epocas = 5

for epoca in range(epocas):
    model.train()
    perdida_acumulada = 0.0
    for imagenes, etiquetas in train_loader:
        optimizer.zero_grad()            # limpiamos los gradientes del paso anterior
        salidas = model(imagenes)        # forward
        loss = criterion(salidas, etiquetas)
        loss.backward()                  # backward: calcula los gradientes
        optimizer.step()                 # actualiza los pesos del modelo
        perdida_acumulada += loss.item()

    perdida_media = perdida_acumulada / len(train_loader)
    print(f'Época {epoca + 1}/{epocas} - pérdida media: {perdida_media:.4f}')

## Parte 6 — Analizzare i risultati

Traccia la perdita e l'accuratezza nel tempo. Cosa noti?

- La CNN è più veloce di una rete fully connected?
- È più precisa?
- C'è sovradattamento (overfitting)?
- Il learning rate è appropriato? (una perdita che salta intorno significa LR troppo alto; una che non scende significa LR troppo basso)

In [ ]:
model.eval()
correctas = 0
total = 0

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        salidas = model(imagenes)
        predicciones = torch.argmax(salidas, dim=1)
        correctas += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

print(f'Exactitud de la CNN en el conjunto de prueba: {correctas / total:.2%}')

## Parte 7 — Salvare il modello addestrato

Dopo l'addestramento, salva i pesi della CNN per usarli in seguito.

In [ ]:
parametros_conv1 = sum(p.numel() for p in model.conv1.parameters())
parametros_densa_equivalente = 3072 * 256

print(f'Parámetros de la primera capa Conv2d de la CNN: {parametros_conv1:,}')
print(f'Parámetros de la primera capa de una red densa equivalente: {parametros_densa_equivalente:,}')
print(f'La capa densa necesitaría {parametros_densa_equivalente / parametros_conv1:.0f} veces más parámetros')

## Riepilogo della lezione

- L'accuratezza (% di previsioni corrette) è spesso più intuitiva della perdita per la classificazione
- Le CNN convergono più velocemente delle reti fully connected su immagini, grazie al riutilizzo di peso (pesi condivisi)
- Se il learning rate è troppo alto, la perdita salta intorno; se è troppo basso, non scende
- Sempre monitorare sia la perdita di addestramento che quella di validazione per rilevare il sovradattamento
- Salva il modello al termine dell'addestramento con `.state_dict()`

Nel notebook di pratica sperimenterai con diversi learning rate e architetture CNN per massimizzare l'accuratezza.